In [1]:
%%writefile parallel_sort.cpp
#include <iostream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>
using namespace std;

// --------------------------------------------------
// Utility functions
// --------------------------------------------------
void printArray(const vector<int>& arr, int limit = 20) {
    for (int i = 0; i < min((int)arr.size(), limit); i++) {
        cout << arr[i] << " ";
    }
    cout << "\n";
}

bool isSorted(const vector<int>& arr) {
    for (int i = 1; i < arr.size(); i++) {
        if (arr[i] < arr[i - 1]) return false;
    }
    return true;
}

// --------------------------------------------------
// Sequential Bubble Sort
// --------------------------------------------------
void sequentialBubbleSort(vector<int>& arr) {
    int n = arr.size();
    for (int i = 0; i < n - 1; i++) {
        bool swapped = false;
        for (int j = 0; j < n - i - 1; j++) {
            if (arr[j] > arr[j + 1]) {
                swap(arr[j], arr[j + 1]);
                swapped = true;
            }
        }
        if (!swapped) break;
    }
}

// --------------------------------------------------
// Parallel Bubble Sort (Odd-Even Transposition)
// --------------------------------------------------
void parallelBubbleSort(vector<int>& arr) {
    int n = arr.size();

    for (int phase = 0; phase < n; phase++) {
        if (phase % 2 == 0) {
            // Even phase
            #pragma omp parallel for
            for (int i = 1; i < n; i += 2) {
                if (arr[i - 1] > arr[i]) {
                    swap(arr[i - 1], arr[i]);
                }
            }
        } else {
            // Odd phase
            #pragma omp parallel for
            for (int i = 1; i < n - 1; i += 2) {
                if (arr[i] > arr[i + 1]) {
                    swap(arr[i], arr[i + 1]);
                }
            }
        }
    }
}

// --------------------------------------------------
// Merge function
// --------------------------------------------------
void merge(vector<int>& arr, int left, int mid, int right) {
    int n1 = mid - left + 1;
    int n2 = right - mid;

    vector<int> L(n1), R(n2);

    for (int i = 0; i < n1; i++) L[i] = arr[left + i];
    for (int i = 0; i < n2; i++) R[i] = arr[mid + 1 + i];

    int i = 0, j = 0, k = left;

    while (i < n1 && j < n2) {
        if (L[i] <= R[j]) arr[k++] = L[i++];
        else arr[k++] = R[j++];
    }

    while (i < n1) arr[k++] = L[i++];
    while (j < n2) arr[k++] = R[j++];
}

// --------------------------------------------------
// Sequential Merge Sort
// --------------------------------------------------
void sequentialMergeSort(vector<int>& arr, int left, int right) {
    if (left >= right) return;

    int mid = left + (right - left) / 2;
    sequentialMergeSort(arr, left, mid);
    sequentialMergeSort(arr, mid + 1, right);
    merge(arr, left, mid, right);
}

// --------------------------------------------------
// Parallel Merge Sort
// depth controls recursion to avoid too many threads
// --------------------------------------------------
void parallelMergeSort(vector<int>& arr, int left, int right, int depth = 0) {
    if (left >= right) return;

    int mid = left + (right - left) / 2;

    if (depth <= 3) {
        #pragma omp parallel sections
        {
            #pragma omp section
            {
                parallelMergeSort(arr, left, mid, depth + 1);
            }

            #pragma omp section
            {
                parallelMergeSort(arr, mid + 1, right, depth + 1);
            }
        }
    } else {
        sequentialMergeSort(arr, left, mid);
        sequentialMergeSort(arr, mid + 1, right);
    }

    merge(arr, left, mid, right);
}

// --------------------------------------------------
// Main
// --------------------------------------------------
int main() {
    int n;
    cout << "Enter number of elements: ";
    cin >> n;

    vector<int> original(n);

    srand(time(0));
    for (int i = 0; i < n; i++) {
        original[i] = rand() % 10000;
    }

    cout << "\nFirst few unsorted elements:\n";
    printArray(original);

    vector<int> a1 = original;
    vector<int> a2 = original;
    vector<int> a3 = original;
    vector<int> a4 = original;

    double start, end;

    // Sequential Bubble Sort
    start = omp_get_wtime();
    sequentialBubbleSort(a1);
    end = omp_get_wtime();
    double seqBubbleTime = end - start;

    // Parallel Bubble Sort
    start = omp_get_wtime();
    parallelBubbleSort(a2);
    end = omp_get_wtime();
    double parBubbleTime = end - start;

    // Sequential Merge Sort
    start = omp_get_wtime();
    sequentialMergeSort(a3, 0, n - 1);
    end = omp_get_wtime();
    double seqMergeTime = end - start;

    // Parallel Merge Sort
    start = omp_get_wtime();
    parallelMergeSort(a4, 0, n - 1);
    end = omp_get_wtime();
    double parMergeTime = end - start;

    // Results
    cout << "\nFirst few sorted elements (Sequential Bubble):\n";
    printArray(a1);

    cout << "\nFirst few sorted elements (Parallel Bubble):\n";
    printArray(a2);

    cout << "\nFirst few sorted elements (Sequential Merge):\n";
    printArray(a3);

    cout << "\nFirst few sorted elements (Parallel Merge):\n";
    printArray(a4);

    cout << "\nSorted check:\n";
    cout << "Sequential Bubble: " << (isSorted(a1) ? "Yes" : "No") << "\n";
    cout << "Parallel Bubble  : " << (isSorted(a2) ? "Yes" : "No") << "\n";
    cout << "Sequential Merge : " << (isSorted(a3) ? "Yes" : "No") << "\n";
    cout << "Parallel Merge   : " << (isSorted(a4) ? "Yes" : "No") << "\n";

    cout << "\nPerformance Comparison:\n";
    cout << "Sequential Bubble Sort Time = " << seqBubbleTime << " seconds\n";
    cout << "Parallel Bubble Sort Time   = " << parBubbleTime << " seconds\n";
    cout << "Sequential Merge Sort Time  = " << seqMergeTime << " seconds\n";
    cout << "Parallel Merge Sort Time    = " << parMergeTime << " seconds\n";

    cout << "\nSpeedup:\n";
    cout << "Bubble Sort Speedup = " << (seqBubbleTime / parBubbleTime) << "\n";
    cout << "Merge Sort Speedup  = " << (seqMergeTime / parMergeTime) << "\n";

    return 0;
}

Writing parallel_sort.cpp


In [2]:
!g++ -fopenmp parallel_sort.cpp -o parallel_sort

In [3]:
!echo 2000 | ./parallel_sort

Enter number of elements: 
First few unsorted elements:
5927 4576 4597 8450 97 9539 6079 4265 9516 362 7140 6209 9426 4916 5430 1621 5724 4721 2736 1527 

First few sorted elements (Sequential Bubble):
3 9 28 34 36 43 58 66 80 97 100 101 105 111 119 119 128 130 136 140 

First few sorted elements (Parallel Bubble):
3 9 28 34 36 43 58 66 80 97 100 101 105 111 119 119 128 130 136 140 

First few sorted elements (Sequential Merge):
3 9 28 34 36 43 58 66 80 97 100 101 105 111 119 119 128 130 136 140 

First few sorted elements (Parallel Merge):
3 9 28 34 36 43 58 66 80 97 100 101 105 111 119 119 128 130 136 140 

Sorted check:
Sequential Bubble: Yes
Parallel Bubble  : Yes
Sequential Merge : Yes
Parallel Merge   : Yes

Performance Comparison:
Sequential Bubble Sort Time = 0.0180213 seconds
Parallel Bubble Sort Time   = 0.0203669 seconds
Sequential Merge Sort Time  = 0.00124195 seconds
Parallel Merge Sort Time    = 0.00081823 seconds

Speedup:
Bubble Sort Speedup = 0.88483
Merge Sort Speedup